## Set up a Redis connection

In [1]:
import os
import warnings
from redis import Redis

warnings.filterwarnings('ignore')

REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = os.getenv("REDIS_PORT", "6379")
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

REDIS_URL = f"redis://:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"

redis_connection = Redis.from_url(REDIS_URL)
redis_connection.ping()

True

## Import the movie data

In [2]:
import pandas as pd

df = pd.read_json("data/movies.json")
print("Loaded", len(df), "movie entries")

df.head()

Loaded 55 movie entries


,id,title,genre,rating,description
0,1,Explosive Pursuit,action,7,A daring cop chases a notorious criminal acros...
1,2,Skyfall,action,8,James Bond returns to track down a dangerous n...
2,3,Fast & Furious 9,action,6,Dom and his crew face off against a high-tech ...
3,4,Black Widow,action,7,Natasha Romanoff confronts her dark past and f...
4,5,John Wick,action,8,A retired hitman seeks vengeance against those...


## Vectorize the data

First, we’ll transform our raw JSON data into vector embeddings, and then we’ll take those embeddings and index them in Redis, making them searchable.

### Creating embeddings using RedisVL

The RedisVL library provides helpful methods to create and manage vector embeddings, making it easier to integrate them into AI workflows. Below, we’ll walk through how to embed a list of movie descriptions using a transformer model and store the results in a DataFrame for later indexing.

There are two key classes that we will use from RedisVL:

`HFTextVectorizer`: This is a wrapper around Hugging Face transformer models that makes it easy to generate vector embeddings.

`EmbeddingsCache`: This class helps store and reuse embeddings using Redis as a cache.

In [3]:
from redisvl.utils.vectorize import HFTextVectorizer
from redisvl.extensions.cache.embeddings import EmbeddingsCache

os.environ["TOKENIZERS_PARALLELISM"] = "false"


hf = HFTextVectorizer(
    model="sentence-transformers/all-MiniLM-L6-v2",
    cache=EmbeddingsCache(
        name="embedcache",
        ttl=600,
        redis_client=redis_connection,
    )
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13216.67it/s]


#### What does the "TOKENIZERS_PARALLELISM" setting do?

This setting is used to clean up the output we receive. It suppresses a warning from the Hugging Face transformers library related to tokenizer parallelism. By default, tokenizers may try to use multiple threads, which can cause warnings or unnecessary overhead in a notebook environment.


#### For HFTextVectorizer we provided two arguments:
* model: The name of a sentence transformer model from Hugging Face. In our case, we’re going to use "sentence-transformers/all-MiniLM-L6-v2", a fast and lightweight model ideal for semantic search.
* cache): An optional argument that assigns an instance of `EmbeddingsCache` that helps avoid recomputing embeddings for repeated inputs.

#### For EmbeddingsCache we provided two arguments:

* name: A key prefix to use in Redis for the cache
* ttl: An optional argument that sets how long the cached embeddings should live in Redis (in seconds)
* redis_client: The Redis client used to store and retrieve vectors.

In [4]:
df["vector"] = hf.embed_many(df["description"].tolist(), as_buffer=True)
df.head()

,id,title,genre,rating,description,vector
0,1,Explosive Pursuit,action,7,A daring cop chases a notorious criminal acros...,b'\x96f|=\xd5`\n;\xf8\x91\xb7;8\xcb~\xbdQe\xce...
1,2,Skyfall,action,8,James Bond returns to track down a dangerous n...,b'\x9aD\x9e\xbd;\x9b\x89\xbc\xb4\x16\x95\xbc}\...
2,3,Fast & Furious 9,action,6,Dom and his crew face off against a high-tech ...,"b'\x17\xa5\xc7\xbc\xef,\xa2=\x12\x19H\xbcU\xc6..."
3,4,Black Widow,action,7,Natasha Romanoff confronts her dark past and f...,b's\xeb\x85\xbd\x13\xcdo\xbd\xcc\xe8\xc2\xbbF\...
4,5,John Wick,action,8,A retired hitman seeks vengeance against those...,b'\xc9;x\xbb\xf8.\xc5=\x9c\x85:;\xc5\xd0\x94<`...


encoded as compact binary buffers (specified by as_buffer=True), in a new column (in the same DataFrame) called "vector"

#### What is a compact binary buffer?

A compact binary buffer is a space-efficient way to store data. In this case, vector embeddings are stored as raw bytes instead of a list of floating-point numbers (e.g.,  b'\x00\x00\x80?\x9a\x99\x19@...' instead of [1.0, 6.4, ...]).
The buffer uses the dtype argument specified in the vectorizer, which defaults to "float32". Since we wanted to keep it as the default, we didn’t need to set it explicitly in our call above.

### Schema creation overview

n core Redis, the definition of a schema and the creation of an index are performed together using the FT.CREATE command. For example, if we were working with a dataset for a company knowledge base, we could run the following command:

```
FT.CREATE knowledge-base
ON JSON 
    PREFIX 1 kb:
SCHEMA
    "$.author" AS "author" TEXT NOSTEM WEIGHT 1.0
    "$.department" AS "department" TAG 
```

In RedisVL, the same underlying FT.CREATE command is executed under the hood, but the process is broken into two separate steps:

Defining the schema
Creating the index based on that schema
We will begin this section by first defining the schema.

The schema can be created by using the IndexSchema class, which lets you define the schema from either a YAML file or a Python dictionary. In this lab, we’ll define one using a dictionary by leveraging the .from_dict() method.

The schema has two main parts that must be defined:

An index section, which configures index-specific settings such as the index name, key prefix, and whether documents are stored as Hashes or JSON.

A fields list, which defines what fields should be indexed and how they should be searched. Each field has a name, a type (e.g., text, tag, numeric, or vector), and any required/optional attributes. A full list of supported field types and attributes can be found in the RedisVL documentation.

You'll build the full schema for the movie data shortly, but to familiarize yourself with the syntax, examine the partial example below before proceeding.

```
schema = IndexSchema.from_dict({
  "index": {
    "name": "movies",
    "prefix": "...",
    "storage_type": "..."
  },
  "fields": [
    {
        "name": "title",
        "type": "text",
        "attrs": {
          ...
        }
    },
    {
      ...
    }
  ]
})
```

### Define the Index Schema

It's time to create the schema. Complete the task below.

#### 📌 Task 1: Create the schema

We've provided the starting code for the movie data schema. Your task is to complete it using the field definitions listed below.

Create a schema named "movies" with:

* A prefix of "movies"
* A storage type of "hash"

Include the following fields:

| Field Name	| Type	| Attributes | 
| --- | --- | --- |
| title	| text	| none
| description | text | none
| genre	| tag |	sortable: True
| rating |	numeric	| sortable: True
| vector |	vector |	dims: 384 distance_metric: "cosine" algorithm: "flat" datatype: "float32"


In [5]:
from redisvl.schema import IndexSchema

index_name = "movies"

schema = IndexSchema.from_dict({
    "index": {
        "name": "movies",
        "prefix": "movies",
        "storage_type": "hash"
    },
    "fields": [
        {
            "name": "title",
            "type": "text"
        },
        {
            "name": "description",
            "type": "text"
        },
        {
            "name": "genre",
            "type": "tag",
            "attrs": {
                "sortable": True
            }
        },
        {
            "name": "rating",
            "type": "numeric",
            "attrs": {
                "sortable": True
            }
        },
        {
            "name": "vector",
            "type": "vector",
            "attrs": {
                "dims": 384,
                "distance_metric": "cosine",
                "algorithm": "flat",
                "datatype": "float32"
            }
        }
    ]
})

print('Schema created:', schema)

Schema created: index=IndexInfo(name='movies', prefix='movies', key_separator=':', storage_type=<StorageType.HASH: 'hash'>, stopwords=None) fields={'title': TextField(name='title', type=<FieldTypes.TEXT: 'text'>, path=None, attrs=TextFieldAttributes(sortable=False, index_missing=False, no_index=False, weight=1, no_stem=False, withsuffixtrie=False, phonetic_matcher=None, index_empty=False, unf=False)), 'description': TextField(name='description', type=<FieldTypes.TEXT: 'text'>, path=None, attrs=TextFieldAttributes(sortable=False, index_missing=False, no_index=False, weight=1, no_stem=False, withsuffixtrie=False, phonetic_matcher=None, index_empty=False, unf=False)), 'genre': TagField(name='genre', type=<FieldTypes.TAG: 'tag'>, path=None, attrs=TagFieldAttributes(sortable=True, index_missing=False, no_index=False, separator=',', case_sensitive=False, withsuffixtrie=False, index_empty=False)), 'rating': NumericField(name='rating', type=<FieldTypes.NUMERIC: 'numeric'>, path=None, attrs=Num

### Add an index

Along with the schema, we need to create the index. To do so, we can use the SearchIndex class, which has a .create() method. The class takes two required arguments:

* schema: The index schema that you want to use.
* client: A Redis client instance.

In [6]:
from redisvl.index import SearchIndex

index = SearchIndex(schema, redis_connection)
index.create(overwrite=True, drop=True)

🚨 Extreme Caution: It is important to be aware when using the `.create()` method, `overwrite=True` overwrites an index in Redis without dropping associated data. However, setting `drop=True` will delete both the index AND all associated documents (clean slate). `drop=True` will only take effect when `overwrite=True` is also set.

Be very careful when setting this, as a mistake may cause data loss.

We can use `rvl`, the CLI built into RedisVL, to confirm that the index was created. Run the code below to view and confirm the movies index was created.

In [7]:
!rvl index info -i movies -u {REDIS_URL}



Index Information:
╭───────────────┬───────────────┬───────────────┬───────────────┬───────────────╮
│ Index Name    │ Storage Type  │ Prefixes      │ Index Options │ Indexing      │
├───────────────┼───────────────┼───────────────┼───────────────┼───────────────┤
| movies        | HASH          | ['movies']    | []            | 0             |
╰───────────────┴───────────────┴───────────────┴───────────────┴───────────────╯
Index Fields:
╭─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────╮
│ Name            │ Attribute       │ Type            │ Field Option    │ Option Value    │ Field Option    │ Option Value    │ Field Option    │ Option Value    │ Field Option    │ Option Value    │
├─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼──────────

### Populate the index

Now, we can load our data into Redis using .load() method. This method has one required argument (and several optional ones) called data which takes an iterable to load (such as our DataFrame rows). When called, it will write each record to Redis, and construct keys using the index prefix, ensuring the data is indexed automatically according to the schema.

In [8]:
index.load(df.to_dict(orient="records"))

['movies:01M4SY0FHWR6K3CKMME62QSS16',
 'movies:01M4SY0FHWR6K3CKMME62QSS17',
 'movies:01M4SY0FHWR6K3CKMME62QSS18',
 'movies:01M4SY0FHWR6K3CKMME62QSS19',
 'movies:01M4SY0FHWR6K3CKMME62QSS1A',
 'movies:01M4SY0FHWR6K3CKMME62QSS1B',
 'movies:01M4SY0FHWR6K3CKMME62QSS1C',
 'movies:01M4SY0FHWR6K3CKMME62QSS1D',
 'movies:01M4SY0FHWR6K3CKMME62QSS1E',
 'movies:01M4SY0FHWR6K3CKMME62QSS1F',
 'movies:01M4SY0FHWR6K3CKMME62QSS1G',
 'movies:01M4SY0FHWR6K3CKMME62QSS1H',
 'movies:01M4SY0FHWR6K3CKMME62QSS1J',
 'movies:01M4SY0FHWR6K3CKMME62QSS1K',
 'movies:01M4SY0FHWR6K3CKMME62QSS1M',
 'movies:01M4SY0FHWR6K3CKMME62QSS1N',
 'movies:01M4SY0FHWR6K3CKMME62QSS1P',
 'movies:01M4SY0FHWR6K3CKMME62QSS1Q',
 'movies:01M4SY0FHWR6K3CKMME62QSS1R',
 'movies:01M4SY0FHWR6K3CKMME62QSS1S',
 'movies:01M4SY0FHWR6K3CKMME62QSS1T',
 'movies:01M4SY0FHWR6K3CKMME62QSS1V',
 'movies:01M4SY0FHWR6K3CKMME62QSS1W',
 'movies:01M4SY0FHWR6K3CKMME62QSS1X',
 'movies:01M4SY0FHWR6K3CKMME62QSS1Y',
 'movies:01M4SY0FHWR6K3CKMME62QSS1Z',
 'movies:01M

### 📌 Task 2: Confirm the data

Confirm the data by fetching one of the movies in the database. Grab any of the keys that were loaded above (e.g., 'movies:01K0791...') and plug it into the code below. Then run it to confirm that it is returned and has a vector field. Alternatively, the data can be viewed in Redis Insight.

In [15]:
# Ex. redis_connection.hgetall("movies:01K07...")

redis_connection.hgetall("movies:01M4SY0FHWR6K3CKMME62QSS2R")

{b'title': b'Ford v Ferrari',
 b'rating': b'8',
 b'genre': b'Action',
 b'vector': b'c0\x91\xbc\xc5e\x8c=.1\x16<\xa5\xd2\x11\xbc#&\xcd\xbdL\xc4\x19=\x10\xab\xb2<\x9f00;\xf5\xdf\xfc\xbc\xabR0\xbde\xcd\x8f\xbc\x0f\xa9J=\xd2xr<\xea@{\xbdm\xe1\xdf\xbd\xe6^\xf1\xbam\x03d<\x94\xf50=\x88\x12A\xbd\x19\xec\xd2<\x1f?\x8c\xbdT\'\xd2\xbd\xa5\xab\xa6\xba!\xe0\xcf=b~\x13\xbc\xe1\xd6\x83=5<\xfa<"\xa2\xcb<\xa9\xaf\xf8\xbc\xdc\x82\r\xbc<}V\xbd\xc4<D\xbdD\x15Y\xbc\xc1\n\xc4<K\xbb9<\xbb\xde\xa9\xbd\xbd\xe9[=9@\x08>\xb6m\xe5<t\x93\xb8\xbd\x93U\xce\xbc\xafK\xcd\xbd\x88>\xa7\xbc\xc3C\x8e<\xe9\x14^=>\xf8\x9b\xbc\xf4_b="\x00-<c3\\;\xa0\xb2\xa0\xbd\xd2\xd9J\xbdN)`\xbc\x86\xc9\xc0<\xba\xb1\x05\xbe\n\xc2Z=>b\xac=\xd0\x94\x86<\x1b\x99\x98={8d=n>,\xbc\xa3\x90\x17=+\xf3Q\xbdB\xd4*<\xb5\xe5\xd3<\xea\xdaK\xbd\x19\x032\xbbDS\x83\xbd\xcb\t\xec<\xf5\x00\x04\xbd\xdb\xd6\xde<\x8c\xa6\xa0=\xe9\xaey\xbdG\xc4\x85\xbc\x99W\x9e\xbd\xb9*\x8e=\x80\xe2:= rn=\xf4\xef\x80\xbbQ\x19\x93=clH;\x83\x9e\x12\xbb3|\xba\xba^8\xc4\xbc:\x99f\x

### Populate the index

Now, we can load our data into Redis using `.load()` method. This method has one required argument (and several optional ones) called data which takes an iterable to load (such as our DataFrame rows). When called, it will write each record to Redis, and construct keys using the index prefix, ensuring the data is indexed automatically according to the schema.

In [10]:
index.load(df.to_dict(orient="records"))

['movies:01M4SY0FJDATWPNXZF2HDXACHM',
 'movies:01M4SY0FJDATWPNXZF2HDXACHN',
 'movies:01M4SY0FJDATWPNXZF2HDXACHP',
 'movies:01M4SY0FJDATWPNXZF2HDXACHQ',
 'movies:01M4SY0FJDATWPNXZF2HDXACHR',
 'movies:01M4SY0FJDATWPNXZF2HDXACHS',
 'movies:01M4SY0FJDATWPNXZF2HDXACHT',
 'movies:01M4SY0FJDATWPNXZF2HDXACHV',
 'movies:01M4SY0FJDATWPNXZF2HDXACHW',
 'movies:01M4SY0FJDATWPNXZF2HDXACHX',
 'movies:01M4SY0FJDATWPNXZF2HDXACHY',
 'movies:01M4SY0FJDATWPNXZF2HDXACHZ',
 'movies:01M4SY0FJDATWPNXZF2HDXACJ0',
 'movies:01M4SY0FJDATWPNXZF2HDXACJ1',
 'movies:01M4SY0FJDATWPNXZF2HDXACJ2',
 'movies:01M4SY0FJDATWPNXZF2HDXACJ3',
 'movies:01M4SY0FJDATWPNXZF2HDXACJ4',
 'movies:01M4SY0FJDATWPNXZF2HDXACJ5',
 'movies:01M4SY0FJDATWPNXZF2HDXACJ6',
 'movies:01M4SY0FJDATWPNXZF2HDXACJ7',
 'movies:01M4SY0FJDATWPNXZF2HDXACJ8',
 'movies:01M4SY0FJDATWPNXZF2HDXACJ9',
 'movies:01M4SY0FJDATWPNXZF2HDXACJA',
 'movies:01M4SY0FJDATWPNXZF2HDXACJB',
 'movies:01M4SY0FJDATWPNXZF2HDXACJC',
 'movies:01M4SY0FJDATWPNXZF2HDXACJD',
 'movies:01M

### What does orient="records" do?

The orient property determines the type of the values of the dictionary. Passing "records" as the value for the orient property makes it so the data has the following structure:
* Each dictionary represents one row (or document)
* The keys are the column names, and the values are the corresponding cell values

### 📌 Task 2: Confirm the data

Confirm the data by fetching one of the movies in the database. Grab any of the keys that were loaded above (e.g., 'movies:01K0791...') and plug it into the code below. Then run it to confirm that it is returned and has a vector field. Alternatively, the data can be viewed in Redis Insight.

In [16]:
# Ex. redis_connection.hgetall("movies:01K07...")

redis_connection.hgetall("movies:01M4SY0FHWR6K3CKMME62QSS2R")

{b'title': b'Ford v Ferrari',
 b'rating': b'8',
 b'genre': b'Action',
 b'vector': b'c0\x91\xbc\xc5e\x8c=.1\x16<\xa5\xd2\x11\xbc#&\xcd\xbdL\xc4\x19=\x10\xab\xb2<\x9f00;\xf5\xdf\xfc\xbc\xabR0\xbde\xcd\x8f\xbc\x0f\xa9J=\xd2xr<\xea@{\xbdm\xe1\xdf\xbd\xe6^\xf1\xbam\x03d<\x94\xf50=\x88\x12A\xbd\x19\xec\xd2<\x1f?\x8c\xbdT\'\xd2\xbd\xa5\xab\xa6\xba!\xe0\xcf=b~\x13\xbc\xe1\xd6\x83=5<\xfa<"\xa2\xcb<\xa9\xaf\xf8\xbc\xdc\x82\r\xbc<}V\xbd\xc4<D\xbdD\x15Y\xbc\xc1\n\xc4<K\xbb9<\xbb\xde\xa9\xbd\xbd\xe9[=9@\x08>\xb6m\xe5<t\x93\xb8\xbd\x93U\xce\xbc\xafK\xcd\xbd\x88>\xa7\xbc\xc3C\x8e<\xe9\x14^=>\xf8\x9b\xbc\xf4_b="\x00-<c3\\;\xa0\xb2\xa0\xbd\xd2\xd9J\xbdN)`\xbc\x86\xc9\xc0<\xba\xb1\x05\xbe\n\xc2Z=>b\xac=\xd0\x94\x86<\x1b\x99\x98={8d=n>,\xbc\xa3\x90\x17=+\xf3Q\xbdB\xd4*<\xb5\xe5\xd3<\xea\xdaK\xbd\x19\x032\xbbDS\x83\xbd\xcb\t\xec<\xf5\x00\x04\xbd\xdb\xd6\xde<\x8c\xa6\xa0=\xe9\xaey\xbdG\xc4\x85\xbc\x99W\x9e\xbd\xb9*\x8e=\x80\xe2:= rn=\xf4\xef\x80\xbbQ\x19\x93=clH;\x83\x9e\x12\xbb3|\xba\xba^8\xc4\xbc:\x99f\x

### Search techniques

In this section, we’ll perform a variety of search techniques on the data we loaded. This includes:

* KNN (K-Nearest Neighbors) search
* Vector search with filters
* Range queries
* Hybrid search

### KNN search

Let's start with the most basic form of vector search that implements the K-Nearest Neighbors (KNN) algorithm. This type of search finds the items in a dataset that are closest in meaning to a given query, even if they are not all relevant.

First, we'll need to embed a specific user query. For now, we'll hardcode this and use `"High tech and action packed movie"`. We can leverage the vectorizer we created earlier `(hf)` and call the `.embed()` method (earlier we used the alternative `.embed_many())`.

In [12]:
user_query = "High tech and action packed movie"

embedded_user_query = hf.embed(user_query)

Now, we can perform the actual search. To do so, we can use the RedisVL `VectorQuery` class. Below, we've defined a query that will get the three closest matches and return the `"title"` and `"genre"` fields.



In [25]:
from redisvl.query import VectorQuery

def vec_query(embedded_user_query):
    return VectorQuery(
    vector=embedded_user_query,
    vector_field_name="vector",
    num_results=3,
    return_fields=["title", "genre"],
    return_score=True,
)

vec_qry = vec_query(embedded_user_query)

print("Vector query:", vec_qry)
print(type(vec_qry))

Vector query: *=>[KNN 3 @vector $vector AS vector_distance] RETURN 3 title genre vector_distance SORTBY vector_distance ASC DIALECT 2 LIMIT 0 3
<class 'redisvl.query.query.VectorQuery'>


#### VectorQuery parameter breakdown

Above, we defined VectorQuery with the following parameters:

* vector: The embedded version of the user's query (we generated this in the previous step, and stored it in the embedded_user_query variable).
* vector_field_name: The name of the vector field in the index (we used "vector" in the schema).
* num_results: The number of closest matches to return — this is the "K" in KNN.
* return_fields: An optional argument that lists the document fields to return from Redis (e.g., "title", "genre"). By default, it will return all the fields.
* return_score: An optional argument that includes the similarity score for each result (lower means more similar). This defaults to False if not set and will not return the score.


Notice that the resulting `vec_query` is a low-level Redis query expression. This is because, in the next step, RedisVL will use the `FT.SEARCH` command under the hood to execute it.

Now that we’ve constructed the query object, we can pass it to `index.query()` to perform the search. Wrapping the results with `pd.DataFrame()` will format the result as a readable table. Run the code below to see it in action!


In [26]:
result = index.query(vec_qry)
pd.DataFrame(result)

,id,vector_distance,title,genre
0,movies:01M4SY0FHWR6K3CKMME62QSS18,0.64973795414,Fast & Furious 9,action
1,movies:01M4SY0FJDATWPNXZF2HDXACHP,0.64973795414,Fast & Furious 9,action
2,movies:01M4SY0FHWR6K3CKMME62QSS29,0.731304764748,No Country for Old Men,Crime


In [28]:
# Example 2:

user_query = "Wholesome animated story about friendship"
embedded_user_query = hf.embed(user_query)

vec_qry = vec_query(embedded_user_query)
print("Vector query:", vec_qry)

result = index.query(vec_qry)
pd.DataFrame(result)

Vector query: *=>[KNN 3 @vector $vector AS vector_distance] RETURN 3 title genre vector_distance SORTBY vector_distance ASC DIALECT 2 LIMIT 0 3


,id,vector_distance,title,genre
0,movies:01M4SY0FHWR6K3CKMME62QSS2S,0.549866139889,A Beautiful Day in the Neighborhood,Biography
1,movies:01M4SY0FJEZN4JCHAQBW9D226K,0.549866139889,A Beautiful Day in the Neighborhood,Biography
2,movies:01M4SY0FHWR6K3CKMME62QSS1J,0.767102241516,Aladdin,comedy


### Vector search with filters

Redis allows you to combine filter searches on fields within the index object which helps create more specific searches. There are a variety of filters for text, number, tag, and geo fields.

For example, say we want to search for the top three movies specifically in the "action" genre. Since the field is a tag (we specified it as a tag in the index schema), we can use the Tag class combined with the .set_filter() method to filter by its value.

Note that the .set_filter() method is a quick way to set the filter expression parameter of the vector query we defined earlier without rewriting it.

Run the code block below to establish and query by the tag. Assuming the query is still "High tech and action packed movie", the result should contain only the movies that are of the "action" genre. If you changed the query in the earlier code block, the result will be different.

In [31]:
from redisvl.query.filter import Tag

tag_filter = Tag("genre") == "action"

vec_qry.set_filter(tag_filter)

result=index.query(vec_qry)
pd.DataFrame(result)

,id,vector_distance,title,genre
0,movies:01M4SY0FHWR6K3CKMME62QSS1B,0.803748786449,Mad Max: Fury Road,action
1,movies:01M4SY0FJDATWPNXZF2HDXACHS,0.803748786449,Mad Max: Fury Road,action
2,movies:01M4SY0FHWR6K3CKMME62QSS19,0.829249322414,Black Widow,action


### Vector search with text filter

Another example of a common filter is a text filter. For example, let's say we wanted to search for movies that directly mention "criminal mastermind" in the description. We can do so using the `Text` class.

In [32]:
from redisvl.query.filter import Text

text_filter = Text("description") % "criminal mastermind"

vec_query = VectorQuery(
    vector=embedded_user_query,
    vector_field_name="vector",
    num_results=3,
    return_fields=["title", "rating", "genre", "description"],
    return_score=True,
    filter_expression=text_filter
)

result = index.query(vec_query)
pd.DataFrame(result)

,id,vector_distance,title,rating,genre,description
0,movies:01M4SY0FHWR6K3CKMME62QSS1Q,0.810938298702,Despicable Me,7,comedy,When a criminal mastermind uses a trio of orph...
1,movies:01M4SY0FJDATWPNXZF2HDXACJ5,0.810938298702,Despicable Me,7,comedy,When a criminal mastermind uses a trio of orph...
2,movies:01M4SY0FJDATWPNXZF2HDXACHT,0.989918470383,The Dark Knight,9,action,"Batman faces off against the Joker, a criminal..."


### Multiple filters

In addition to using individual filters, we can also filter with a combination of filters. This can't be accomplished by calling .set_filter() multiple times, because it does not merge filters; rather, it overwrites any previous filter expression.

Instead, the combined filter needs to be created ahead of time using logical operators like & (AND) or | (OR), and then passed once to .set_filter() or to the query when it is defined.

In [33]:
from redisvl.query.filter import Num

num_filter = Num("rating") >= 7
tag_filter = Tag("genre") == "action"

combined_filter = tag_filter & num_filter

# rebuild vector query
vec_query = VectorQuery(
    vector=embedded_user_query,
    vector_field_name="vector",
    num_results=3,
    return_fields=["title", "rating", "genre"],
    return_score=True,
    filter_expression=combined_filter
)

result = index.query(vec_query)
pd.DataFrame(result)

,id,vector_distance,title,rating,genre
0,movies:01M4SY0FHWR6K3CKMME62QSS1B,0.803748786449,Mad Max: Fury Road,8,action
1,movies:01M4SY0FJDATWPNXZF2HDXACHS,0.803748786449,Mad Max: Fury Road,8,action
2,movies:01M4SY0FJDATWPNXZF2HDXACHQ,0.829249322414,Black Widow,7,action


### Range queries

So far, we've used K-Nearest Neighbors (KNN) search, which always returns the top N results, even if some aren't very similar. Range queries give us more control by letting us define a cutoff for what counts as "similar enough."

Instead of asking, "What are the closest 5 documents?" a range query lets us say, "Only return documents within a certain distance."

This is useful when you care more about quality than quantity and don’t want to include loosely related results. This distance is defined via a distance_threshold parameter defined on a RangeQuery class instance.

In [34]:
from redisvl.query import RangeQuery

user_query = "Family friendly fantasy movies"

embedded_user_query = hf.embed(user_query)

range_query = RangeQuery(
    vector=embedded_user_query,
    vector_field_name="vector",
    return_fields=["title", "rating", "genre"],
    return_score=True,
    distance_threshold=0.8  # find all items with a semantic distance of less than 0.8
)

result = index.query(range_query)
pd.DataFrame(result)


,id,vector_distance,title,rating,genre
0,movies:01M4SY0FHWR6K3CKMME62QSS1N,0.64470243454,The Incredibles,8,comedy
1,movies:01M4SY0FJDATWPNXZF2HDXACJ3,0.64470243454,The Incredibles,8,comedy
2,movies:01M4SY0FHWR6K3CKMME62QSS2W,0.727557063103,Knives Out,8,Mystery
3,movies:01M4SY0FJEZN4JCHAQBW9D226P,0.727557063103,Knives Out,8,Mystery
4,movies:01M4SY0FHWR6K3CKMME62QSS19,0.747986912727,Black Widow,7,action
5,movies:01M4SY0FJDATWPNXZF2HDXACHQ,0.747986912727,Black Widow,7,action
6,movies:01M4SY0FHWR6K3CKMME62QSS1Q,0.750915408134,Despicable Me,7,comedy
7,movies:01M4SY0FJDATWPNXZF2HDXACJ5,0.750915408134,Despicable Me,7,comedy
8,movies:01M4SY0FHWR6K3CKMME62QSS1M,0.75129878521,Shrek,8,comedy
9,movies:01M4SY0FJDATWPNXZF2HDXACJ2,0.75129878521,Shrek,8,comedy


### How does distance threshold work?

Distance is interpreted using the value set for "distance_metric" via the index schema. In ours, we defined it as "cosine".

Cosine distance is in the range of 0 to 2 which can be normalized (to be between 0 and 1) if you set an optional normalize_vector_distance parameter to True. The cosine function isn't linear, but as a quick estimate, a value of 0.8 is roughly an 80% miss (i.e. Anything closer than 80% of the maximum possible distance).

### Hybrid search

So far, we’ve explored vector search, which matches based on meaning, and we know that Redis supports searching by text as well (both via a filter and a completely separate TextQuery class for text-only queries, which do not invoke vector similarity). However, what if we want the best of both? That’s where hybrid search comes in.

Hybrid search allows you to combine full-text search with vector similarity to improve relevance and precision.

For example, a hybrid query for "movies about artificial intelligence" might return:

* A lexical match: “Artificial Intelligence: AI” (direct keyword match)
* A Semantic match: “Ex Machina” or “Her” — movies that don’t use the exact phrase but are about the requested topic.
With hybrid search, both types of results appear—sorted by how well they match your intent and keywords.

The great news is that RedisVL has a built-in HybridQuery class to help run these types of queries. It takes both a text input and a query vector, and Redis will return results that reflect both signals.



In [46]:
# from redisvl.query import HybridQuery

# hybrid_query = HybridQuery(
#     text=user_query,
#     text_field_name="description",
#     text_scorer="BM25",
#     vector=embedded_user_query,
#     vector_field_name="vector",
    
#     combination_method="LINEAR",
#     linear_alpha=0.3,  # 30% text, 70% vector
#     yield_text_score_as="text_score",
#     yield_vsim_score_as="vector_score",
#     yield_combined_score_as="hybrid_score",
    
#     num_results=20,
#     return_fields=["title", "description"],
# )

# print("Hybrid query object:", hybrid_query)

In [47]:
# result = index.query(hybrid_query)[:4]
# print(pd.DataFrame(result))
# #pd.DataFrame(result)[["title", "yield_vsim_score_as", "yield_text_score_as", "yield_combined_score_as"]]

In [49]:
from redisvl.query import AggregateHybridQuery
import pandas as pd

hybrid_query = AggregateHybridQuery(
    text=user_query,
    text_field_name="description",
    text_scorer="BM25STD",
    vector=embedded_user_query,
    vector_field_name="vector",
    alpha=0.7,
    num_results=20,
    return_fields=["title", "description"],
)

result = index.query(hybrid_query)[:4]

pd.DataFrame(result)


,vector_distance,title,description,vector_similarity,text_score,hybrid_score
0,0.727557063103,Knives Out,A detective investigates the death of a patria...,0.636221468449,4.99810092901,1.94478530662
1,0.727557063103,Knives Out,A detective investigates the death of a patria...,0.636221468449,4.99810092901,1.94478530662
2,0.752483725548,Marriage Story,An incisive and compassionate look at a marria...,0.623758137226,4.8184893272,1.88217749422
3,0.752483725548,Marriage Story,An incisive and compassionate look at a marria...,0.623758137226,4.8184893272,1.88217749422
